Extracting tweets from Twitter

In [ ]:
import time
import random
import pandas as pd
from ntscraper import Nitter

# Initialize the scraper
scraper = Nitter()

# Function to switch instances if needed
def switch_nitter_instance(scraper, instance_url):
    scraper.base_url = instance_url
    print(f"Switched to Nitter instance: {instance_url}")

# Function to randomize sleep
def random_sleep(min_time=1, max_time=3):
    sleep_time = random.uniform(min_time, max_time)
    time.sleep(sleep_time)

# List of Nitter instances to switch between if one fails
nitter_instances = [
    "https://nitter.net",
    "https://nitter.42l.fr",
    "https://nitter.snopyta.org"
]

# List of major roads in key Pakistani cities (Islamabad, Lahore, Karachi, Multan, etc.)
major_roads = [
    # Islamabad/Rawalpindi
    "Murree Road", "9th Avenue", "IJP Road", "Airport Road", "Saddar Road", "Faizabad", "Kashmir Highway",
    "Margalla Road", "Jinnah Avenue", "Blue Area", "Peshawar Road", "GT Road", "Expressway", "Club Road",
    "Pir Wadhai Road", "Raja Bazaar", "Bahria Town Road",
    # Lahore
    "Ferozepur Road", "Canal Road", "MM Alam Road", "Mall Road", "Jail Road", "Lahore Ring Road",
    "Multan Road", "GT Road", "Defence Road", "Allama Iqbal Road", "College Road", "Raiwind Road",
    "Walton Road", "Bedian Road", "Shahrah-e-Quaid-e-Azam", "Wahdat Road",
    # Karachi
    "Shahrah-e-Faisal", "Rashid Minhas Road", "MA Jinnah Road", "Korangi Road", "University Road",
    "Lyari Expressway", "Karachi-Hyderabad Motorway (M9)", "Karsaz Road", "Sharah-e-Pakistan",
    "Nazimabad Road", "Saddar", "II Chundrigar Road", "Clifton Road", "Kemari Road",
    # Multan
    "Bosan Road", "Abdali Road", "Khanewal Road", "Vehari Road", "Sher Shah Road", "Multan Road",
    "Qadirpur Raan Road", "Old Shujabad Road", "Gulgasht Colony Road", "Chungi Road", "Nishter Road",
    # National Highways
    "Motorway M2", "Motorway M3", "Motorway M4", "GT Road", "N5 National Highway", "Indus Highway",
    "RCD Highway", "Makran Coastal Highway", "Karakoram Highway", "Silk Route"
]

english_keywords = [
    "accident", "collision", "roadblock", "container", "road safety", "road closure", "traffic",
    "blockage", "road hazard", "pile-up", "road accident", "traffic jam"
]

urdu_keywords = [
    "حادثہ", "ٹریفک", "سڑک بند", "سڑک حادثہ", "روڈ بلاک", "ٹریفک جام", "پل بند", "راستہ بند",
    "سڑک ٹوٹ گئی", "سیلاب", "رکاوٹ", "بارش", "پھسلن", "رکاوٹ", "ٹریفک پولیس", "ٹرک", "کنٹینر"
]

hashtags = ["#ChampAlertsOnTheGo", "#CAOTG"]

# Combine all keywords into one list
combined_keywords = major_roads + english_keywords + urdu_keywords + hashtags

# Dictionary to store tweet data
data = {
    'link': [],
    'text': [],
    'user': [],
    'likes': [],
    'retweets': [],
    'comments': []
}

# Log failed keywords for retry later
failed_keywords = []

# Start with the first Nitter instance
current_instance_index = 0
switch_nitter_instance(scraper, nitter_instances[current_instance_index])

# Set the number of tweets to retrieve per keyword
num_of_tweets = 50  # Adjust as necessary

# Geolocation filtering based on cities or regions in Pakistan
target_locations = [
    "Islamabad", "Rawalpindi", "Lahore", "Karachi", "Multan",
    "Peshawar", "Sialkot", "Faisalabad", "Quetta", "Sargodha"
]

# Function to filter tweets by location mentioned in bio or tweet
def geo_filter_tweet(tweet, target_locations):
    # Safely access the user's bio, use an empty string if the bio is missing
    user_bio = tweet['user'].get('bio', '').lower()

    # Safely access the tweet text
    tweet_text = tweet['text'].lower()

    # Check if any target location is mentioned in the bio or tweet text
    for location in target_locations:
        location_lower = location.lower()
        if location_lower in user_bio or location_lower in tweet_text:
            return True
    return False



# Function to avoid duplicates based on tweet URLs
def is_unique_tweet(tweet_link, crawled_links):
    if tweet_link not in crawled_links:
        crawled_links.add(tweet_link)
        return True
    return False

# Track unique tweet links to avoid duplicates
crawled_links = set()

# Loop through combined keywords to find tweets
for keyword in combined_keywords:
    try:
        tweets_data = scraper.get_tweets(keyword, mode='term', number=num_of_tweets)

        if tweets_data and 'tweets' in tweets_data:
            print(f"Found tweets for keyword: {keyword}")

            for tweet in tweets_data['tweets']:
                tweet_link = tweet['link']

                # Check if the tweet matches the geolocation criteria
                if geo_filter_tweet(tweet, target_locations):
                    if is_unique_tweet(tweet_link, crawled_links):  # Ensure no duplicates
                        data['link'].append(tweet_link)
                        data['text'].append(tweet['text'])
                        data['user'].append(tweet['user']['name'])
                        data['likes'].append(tweet['stats']['likes'])
                        data['retweets'].append(tweet['stats']['retweets'])
                        data['comments'].append(tweet['stats']['comments'])
                else:
                    print(f"Tweet did not match target location: {keyword}")
        else:
            print(f"No tweets found for keyword: {keyword}")

    except ValueError as e:
        print(f"Failed for keyword: {keyword} due to instance issues.")
        failed_keywords.append(keyword)

        # Switch to the next instance if available
        current_instance_index += 1
        if current_instance_index < len(nitter_instances):
            switch_nitter_instance(scraper, nitter_instances[current_instance_index])
        else:
            print("No more Nitter instances available. Stopping.")
            break
        random_sleep(5, 10)  # Sleep longer before retrying

    random_sleep()  # Add random sleep to avoid being blocked

# Check if any tweets were collected and save to CSV
if data['text']:
    df = pd.DataFrame(data)
    df.to_csv('road_safety_tweets_pakistanr.csv', index=False)
    print("Tweets saved to road_safety_tweets_pakistan.csv")
else:
    print("No tweets found for any of the provided keywords.")

# Print failed keywords for retrying
if failed_keywords:
    print("Retry the following keywords due to instance issues:", failed_keywords)


Testing instances: 100%|██████████| 16/16 [00:05<00:00,  2.70it/s]
INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Switched to Nitter instance: https://nitter.net


Found tweets for keyword: Murree Road


INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Failed for keyword: 9th Avenue due to instance issues.
Switched to Nitter instance: https://nitter.42l.fr


...

Found tweets for keyword: کنٹینر


INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Found tweets for keyword: #ChampAlertsOnTheGo


INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Found tweets for keyword: #CAOTG
No tweets found for any of the provided keywords.
Retry the following keywords due to instance issues: ['9th Avenue']


In [ ]:
import time
import random
import pandas as pd
from ntscraper import Nitter

# Initialize the scraper
scraper = Nitter()

# Function to switch instances if needed
def switch_nitter_instance(scraper, instance_url):
    scraper.base_url = instance_url
    print(f"Switched to Nitter instance: {instance_url}")

# Function to randomize sleep
def random_sleep(min_time=1, max_time=3):
    sleep_time = random.uniform(min_time, max_time)
    time.sleep(sleep_time)

# List of Nitter instances to switch between if one fails
nitter_instances = [
    "https://nitter.net",
    "https://nitter.42l.fr",
    "https://nitter.snopyta.org",
    "https://nitter.privacydev.net"
]

# Dictionary to store tweet data
data = {
    'link': [],
    'text': [],
    'user': [],
    'likes': [],
    'retweets': [],
    'comments': []
}

# Log failed keywords for retry later
failed_keywords = []

# Start with the first Nitter instance
current_instance_index = 0
switch_nitter_instance(scraper, nitter_instances[current_instance_index])

# Set the number of tweets to retrieve per keyword
num_of_tweets = 50  # Adjust as necessary

# Function to avoid duplicates based on tweet URLs
def is_unique_tweet(tweet_link, crawled_links):
    if tweet_link not in crawled_links:
        crawled_links.add(tweet_link)
        return True
    return False

# Track unique tweet links to avoid duplicates
crawled_links = set()

# Function to filter tweets by location mentioned in bio or tweet
def geo_filter_tweet(tweet, target_locations):
    user_bio = tweet['user'].get('bio', '').lower()
    tweet_text = tweet['text'].lower()

    for location in target_locations:
        location_lower = location.lower()
        if location_lower in user_bio or location_lower in tweet_text:
            return True
    return False

# Geolocation filtering based on cities or regions in Pakistan
target_locations = [
    "Islamabad", "Rawalpindi", "Lahore", "Karachi", "Multan",
    "Peshawar", "Sialkot", "Faisalabad", "Quetta", "Sargodha",
    # Tourist locations
    "Swat", "Gilgit", "Hunza", "Chitral", "Murree", "Muzaffarabad", "Neelum Valley", "Kaghan", "Naran"
]

# Define a retry limit
retry_limit = 3
retry_counter = 0

# Loop through combined keywords to find tweets
for keyword in combined_keywords:
    retry_counter = 0  # Reset retry counter for each keyword

    while retry_counter < retry_limit:
        try:
            tweets_data = scraper.get_tweets(keyword, mode='term', number=num_of_tweets)

            if tweets_data and 'tweets' in tweets_data:
                print(f"Found tweets for keyword: {keyword}")

                for tweet in tweets_data['tweets']:
                    tweet_link = tweet['link']

                    # Check if the tweet matches the geolocation criteria
                    if geo_filter_tweet(tweet, target_locations):
                        if is_unique_tweet(tweet_link, crawled_links):  # Ensure no duplicates
                            data['link'].append(tweet_link)
                            data['text'].append(tweet['text'])
                            data['user'].append(tweet['user']['name'])
                            data['likes'].append(tweet['stats']['likes'])
                            data['retweets'].append(tweet['stats']['retweets'])
                            data['comments'].append(tweet['stats']['comments'])
                    else:
                        print(f"Tweet did not match target location: {keyword}")
                break  # Successfully retrieved tweets, break out of retry loop
            else:
                print(f"No tweets found for keyword: {keyword}")
                break  # No tweets, move to the next keyword

        except Exception as e:
            print(f"Failed for keyword: {keyword} due to {e}. Retrying...")

            # Increment the retry counter
            retry_counter += 1
            if retry_counter >= retry_limit:
                failed_keywords.append(keyword)
                print(f"Failed for keyword: {keyword} after {retry_limit} attempts.")

            # Switch to the next instance if available
            current_instance_index += 1
            if current_instance_index < len(nitter_instances):
                switch_nitter_instance(scraper, nitter_instances[current_instance_index])
            else:
                print("No more Nitter instances available. Stopping.")
                break

            random_sleep(5, 10)  # Sleep longer before retrying

    random_sleep()  # Add random sleep to avoid being blocked

# Check if any tweets were collected and save to CSV
if data['text']:
    df = pd.DataFrame(data)
    df.to_csv('road_safety_tweets_pakistan.csv', index=False)
    print("Tweets saved to road_safety_tweets_pakistan.csv")
else:
    print("No tweets found for any of the provided keywords.")

# Print failed keywords for retrying
if failed_keywords:
    print("Retry the following keywords due to instance issues:", failed_keywords)


extracting relevant information like location severity from tweets uisng llama3b model

In [ ]:
import os
import pandas as pd
from groq import Groq

# Set up the API Key directly
client = Groq(api_key="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")

# Load the CSV file with tweets
file_path = 'road_safety_tweets_pakistan2.csv'  # Use your file path
tweets_df = pd.read_csv(file_path)

# Enhanced function to extract safety issue, area, and severity
def extract_safety_information(tweet):
    prompt = (
        f"Extract the area, safety issue, and severity from this tweet: '{tweet}'. "
        "The safety issue can be an accident, fire, roadblock, or any other incident. "
        "If it's related to traffic or congestion, mark the severity as low. "
        "If there is no clear safety issue, mark it as irrelevant."
    )

    try:
        # Using a different model for potentially better context understanding
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="llama3-8b-8192",  # Switching to Meta Llama 3 8B
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Apply the function to all tweets in the CSV
tweets_df['Extracted_Info'] = tweets_df['text'].apply(extract_safety_information)

# Save the results to a new CSV file
tweets_df.to_csv('1improved_processed_road_safety_tweets.csv', index=False)

# Print the first few rows of processed data
print(tweets_df.head())


                                                link  \
0  https://twitter.com/Hanii_tales/status/1840070...   
1  https://twitter.com/riazghafur/status/18410653...   
2  https://twitter.com/Anonymous469437/status/184...   
3  https://twitter.com/thebeekeeper23/status/1840...   
4  https://twitter.com/saddam_hashmii/status/1840...   

                                                text           user  likes  \
0  People are putting up a strong resistance in R...      Khan Saab    503   
1  Compassion in Action; a traffic policeman assi...    Riaz Ghafur      0   
2  Mai Peshawar se nhi hu lekn muje pta hai.. fir...   Anonymous ❤️      0   
3                   Khaadi Arbab Road Peshawar.... 🙂     Black Burn      1   
4  Dear concern Team 🚨 She belongs to Hassan garh...  saddam hashim      0   

   retweets  comments                                     Extracted_Info  
0       149         3  Here are the extracted information:\n\n* Area:...  
1         0         0  Here are the extracte

scrapping news for our road hazardous keywords

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

# Path to the extracted ChromeDriver in your Downloads folder
chrome_driver_path = r"C:\Users\Dell\Downloads\chromedriver-win64\chromedriver.exe"

# Initialize WebDriver
service = Service(executable_path=chrome_driver_path)
driver = webdriver.Chrome(service=service)

# Open a test page (Google)
driver.get('https://www.google.com')

# Close the browser
driver.quit()


In [ ]:
import csv
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from time import sleep
import random

# Path to your ChromeDriver (adjust it based on your system)
chrome_driver_path = r"C:\Users\Dell\Downloads\chromedriver-win64\chromedriver.exe"

# Set up Selenium Chrome options
options = Options()
options.add_argument("--start-maximized")  # Start the browser maximized
options.add_argument("--disable-infobars")
options.add_argument("--disable-extensions")

# Initialize WebDriver
service = Service(executable_path=chrome_driver_path)
driver = webdriver.Chrome(service=service, options=options)

# Keywords
english_keywords = [
    "accident", "collision", "roadblock", "container", "road safety", "road closure", "traffic",
    "blockage", "road hazard", "pile-up", "road accident", "traffic jam"
]
urdu_keywords = [
    "حادثہ", "ٹریفک", "سڑک بند", "سڑک حادثہ", "روڈ بلاک", "ٹریفک جام", "پل بند", "راستہ بند",
    "سڑک ٹوٹ گئی", "سیلاب", "رکاوٹ", "بارش", "پھسلن", "رکاوٹ", "ٹریفک پولیس", "ٹرک", "کنٹینر"
]
# Major roads in Pakistan
major_roads = [
    # Islamabad/Rawalpindi
    "Murree Road", "9th Avenue", "IJP Road", "Airport Road", "Saddar Road", "Faizabad", "Kashmir Highway",
    "Margalla Road", "Jinnah Avenue", "Blue Area", "Peshawar Road", "GT Road", "Expressway", "Club Road",
    "Pir Wadhai Road", "Raja Bazaar", "Bahria Town Road",
    # Lahore
    "Ferozepur Road", "Canal Road", "MM Alam Road", "Mall Road", "Jail Road", "Lahore Ring Road",
    "Multan Road", "GT Road", "Defence Road", "Allama Iqbal Road", "College Road", "Raiwind Road",
    "Walton Road", "Bedian Road", "Shahrah-e-Quaid-e-Azam", "Wahdat Road",
    # Karachi
    "Shahrah-e-Faisal", "Rashid Minhas Road", "MA Jinnah Road", "Korangi Road", "University Road",
    "Lyari Expressway", "Karachi-Hyderabad Motorway (M9)", "Karsaz Road", "Sharah-e-Pakistan",
    "Nazimabad Road", "Saddar", "II Chundrigar Road", "Clifton Road", "Kemari Road",
    # Multan
    "Bosan Road", "Abdali Road", "Khanewal Road", "Vehari Road", "Sher Shah Road", "Multan Road",
    "Qadirpur Raan Road", "Old Shujabad Road", "Gulgasht Colony Road", "Chungi Road", "Nishter Road",
    # National Highways
    "Motorway M2", "Motorway M3", "Motorway M4", "GT Road", "N5 National Highway", "Indus Highway",
    "RCD Highway", "Makran Coastal Highway", "Karakoram Highway", "Silk Route"
]

# List to store news results
news_results = []

# Limit of news articles per keyword
ARTICLE_LIMIT = 10

# Function to extract news articles along with their posting time
def fetch_news(keyword, road_name):
    search_url = f'https://www.google.com/search?q={keyword}+{road_name}&tbm=nws'
    driver.get(search_url)
    sleep(random.uniform(3, 5))  # Wait for the page to load

    # Get page source and parse with BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    print(f"Processing keyword '{keyword}' on road '{road_name}'")

    # Extract news articles
    articles = soup.find_all('div', class_='SoaBEf')
    print(f"Found {len(articles)} articles for keyword '{keyword}' on '{road_name}'")

    article_count = 0
    for item in articles:
        if article_count >= ARTICLE_LIMIT:
            break

        try:
            # Extract raw link
            link_tag = item.find('a', href=True)
            if link_tag:
                news_link = link_tag['href']

            # Extract title and description
            title_tag = item.find('div', class_='n0jPhd')  # Class for title
            description_tag = item.find('div', class_='GI74Re nDgy9d')  # Class for description

            # Extract time (e.g., minutes/hours ago)
            time_tag = item.find('div', class_='OSrXXb rbYSKb LfVVr')  # Class for the time posted
            if time_tag:
                posted_time = time_tag.find('span').get_text()
            else:
                posted_time = "Time not found"

            if title_tag and description_tag:
                title = title_tag.get_text()
                description = description_tag.get_text()

                # Print extracted information for debugging purposes
                print(f"Title: {title}, Time: {posted_time}, Link: {news_link}")

                # Save the results in the list
                news_results.append([road_name, keyword, title, description, news_link, posted_time])
                article_count += 1

        except Exception as e:
            print(f"Error processing item: {e}")

# Fetch news for each road and keyword (English and Urdu)
for road in major_roads:
    print(f"Fetching news for road: {road} with English keywords")
    for keyword in english_keywords:
        fetch_news(keyword, road)

    print(f"Fetching news for road: {road} with Urdu keywords")
    for keyword in urdu_keywords:
        fetch_news(keyword, road)

# Save the news results into a CSV file
csv_filename = "road_hazard_news.csv"
df = pd.DataFrame(news_results, columns=["Road Name", "Keyword", "Title", "Description", "Link", "Time Posted"])
df.to_csv(csv_filename, index=False, encoding='utf-8')

# Close WebDriver when done
driver.quit()

print(f"News results saved to {csv_filename}")


In [ ]:
# Assuming 'tweets_df' is the DataFrame containing your data

# Specify the file path where you want to save the CSV
output_file_path = 'processed_road_safety_tweets.csv'

# Save the DataFrame to a CSV file
tweets_df.to_csv(output_file_path, index=False)

# Print confirmation
print(f"DataFrame has been saved to {output_file_path}")


DataFrame has been saved to processed_road_safety_tweets.csv


getting news from last 3 days for road hazard

In [ ]:
# Import the necessary library
from google.colab import files

# Upload the file(s) using Colab's file uploader widget
uploaded = files.upload()

# List the uploaded files
for file_name in uploaded.keys():
    print(f"Uploaded file: {file_name}")

Saving road_hazard_news.csv to road_hazard_news.csv
Uploaded file: road_hazard_news.csv


In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import re

# Read the data with the correct separator
df = pd.read_csv('road_hazard_news.csv', sep=',')  # Adjust 'sep' if necessary

# Verify columns
print("Column names:", df.columns)
if 'Time Posted' not in df.columns:
    print("The 'Time Posted' column was not found in the DataFrame.")
    print("Available columns:", df.columns)
    raise KeyError("'Time Posted' column not found.")

# Define the parse_time_posted function
def parse_time_posted(time_str):
    match = re.match(r'(\d+)\s+(\w+)\s+ago', str(time_str))
    if match:
        value, unit = match.groups()
        value = int(value)
        unit = unit.lower()
        if unit in ['day', 'days']:
            delta = timedelta(days=value)
        elif unit in ['hour', 'hours']:
            delta = timedelta(hours=value)
        elif unit in ['minute', 'minutes']:
            delta = timedelta(minutes=value)
        else:
            return pd.NaT
        return datetime.now() - delta
    else:
        try:
            return pd.to_datetime(time_str)
        except:
            return pd.NaT

# Apply the function
df['Parsed Time Posted'] = df['Time Posted'].apply(parse_time_posted)

# Remove rows with NaT in 'Parsed Time Posted'
df = df.dropna(subset=['Parsed Time Posted'])

# Filter entries from the last 3 days
now = datetime.now()
filtered_df = df[df['Parsed Time Posted'] >= (now - timedelta(days=3))]

# Reset index
filtered_df2 = filtered_df.reset_index(drop=True)

# Display the filtered data
print(filtered_df2[['Road Name', 'Keyword', 'Title', 'Time Posted']])


Column names: Index(['Road Name', 'Keyword', 'Title', 'Description', 'Link', 'Time Posted'], dtype='object')
               Road Name       Keyword  \
0            Murree Road     container   
1            Murree Road     container   
2            Murree Road     container   
3            Murree Road  road closure   
4            Murree Road  road closure   
..                   ...           ...   
260              GT Road       traffic   
261              GT Road       traffic   
262              GT Road         ٹریفک   
263  N5 National Highway   road safety   
264           Silk Route     collision   

                                                 Title Time Posted  
0    Containers on roads a nuisance for Islamabad r...  2 days ago  
1    PTI protest: Barriers being removed in twin ci...  2 days ago  
2    PTI protest enters third day: Traffic flows in...  2 days ago  
3              Road closures and openings in Islamabad  2 days ago  
4    Islamabad, Rawalpindi paralyzed for 

getting news from last 24 hours and extarcting relevant information from it

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import re
from groq import Groq

# Set up the Groq client
client = Groq(api_key="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")

# Read the data with the correct separator
df = pd.read_csv('road_hazard_news.csv', sep=',')  # Adjust 'sep' if necessary

# Verify columns
print("Column names:", df.columns)
if 'Time Posted' not in df.columns:
    print("The 'Time Posted' column was not found in the DataFrame.")
    raise KeyError("'Time Posted' column not found.")

# Define the parse_time_posted function
def parse_time_posted(time_str):
    match = re.match(r'(\d+)\s+(\w+)\s+ago', str(time_str))
    if match:
        value, unit = match.groups()
        value = int(value)
        unit = unit.lower()
        if unit in ['day', 'days']:
            delta = timedelta(days=value)
        elif unit in ['hour', 'hours']:
            delta = timedelta(hours=value)
        elif unit in ['minute', 'minutes']:
            delta = timedelta(minutes=value)
        else:
            return pd.NaT
        return datetime.now() - delta
    else:
        try:
            return pd.to_datetime(time_str)
        except:
            return pd.NaT

# Apply the function to parse 'Time Posted'
df['Parsed Time Posted'] = df['Time Posted'].apply(parse_time_posted)

# Remove rows with NaT in 'Parsed Time Posted'
df = df.dropna(subset=['Parsed Time Posted'])

# Filter entries from the last 24 hours
now = datetime.now()
filtered_df = df[df['Parsed Time Posted'] >= (now - timedelta(days=1))]

# Reset the index for the filtered data
filtered_df = filtered_df.reset_index(drop=True)

# Define the Groq function to extract safety information
def extract_safety_information(description):
    prompt = (
        f"Extract the area, safety issue, and severity from this description: '{description}'. "
        "The safety issue can be an accident, fire, roadblock, or any other incident. "
        "If it's related to traffic or congestion, mark the severity as low. "
        "If there is no clear safety issue, mark it as irrelevant."
    )
    try:
        # Using Groq API to generate response
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="llama3-8b-8192"  # Using Meta Llama 3 model
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Apply the Groq function to the 'Description' column in the filtered dataframe
filtered_df['Extracted_Info'] = filtered_df['Description'].apply(extract_safety_information)

# Save the result to a new CSV file
output_file = 'processed_last_3_day_news.csv'
filtered_df.to_csv(output_file, index=False)

# Print the first few rows of the processed data
print(filtered_df.head())


Column names: Index(['Road Name', 'Keyword', 'Title', 'Description', 'Link', 'Time Posted'], dtype='object')
      Road Name       Keyword  \
0  Airport Road  road closure   
1  Airport Road       traffic   
2    Expressway     collision   
3    Expressway     collision   
4    Expressway  road closure   

                                               Title  \
0  Traffic and Travel - Latest news updates, pict...   
1  Traffic and Travel - Latest news updates, pict...   
2  Santa Rosa man charged with murder in crash in...   
3        A38 crash sparks queues on edge of Dartmoor   
4  Highway 144 closed between Cartier and Sultan ...   

                                         Description  \
0  Details of new Belfast city centre to Dublin b...   
1  Details of new Belfast city centre to Dublin b...   
2  Investigators say the defendant intentionally ...   
3  A crash left traffic queuing on the A38 in Dev...   
4  A crash Tuesday afternoon on Highway 144 has c...   

                  

extarcting relevant information from last 24 hours news

In [ ]:
import os
import pandas as pd
from groq import Groq

# Set up the Groq client with the API key
client = Groq(api_key="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")

# Assuming filtered_df2 is the dataframe after filtering the last 3 days
# It should have a 'Description' column or whichever column you want to analyze

# Define the Groq function to extract safety information
def extract_safety_information(description):
    prompt = (
        f"Extract the area, safety issue, and severity from this description: '{description}'. "
        "The safety issue can be an accident, fire, roadblock, or any other incident. "
        "If it's related to traffic or congestion, mark the severity as low. "
        "If there is no clear safety issue, mark it as irrelevant."
    )

    try:
        # Using Groq API
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="llama3-8b-8192"  # Meta Llama 3 model for better context
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Apply the Groq function to the 'Description' column in the filtered dataframe
filtered_df2['Extracted_Info'] = filtered_df2['Description'].apply(extract_safety_information)

# Save the result to a new CSV file
output_file = 'aprocessed_filtered_news.csv'
filtered_df2.to_csv(output_file, index=False)

# Print the first few rows of the processed data
print(filtered_df2.head())


     Road Name       Keyword  \
0  Murree Road     container   
1  Murree Road     container   
2  Murree Road     container   
3  Murree Road  road closure   
4  Murree Road  road closure   

                                               Title  \
0  Containers on roads a nuisance for Islamabad r...   
1  PTI protest: Barriers being removed in twin ci...   
2  PTI protest enters third day: Traffic flows in...   
3            Road closures and openings in Islamabad   
4  Islamabad, Rawalpindi paralyzed for third day ...   

                                         Description  \
0  Caught in the middle of political struggle, th...   
1  The process of removing obstacles in Islamabad...   
2  Islamabad woke up to a tense and quiet morning...   
3  Islamabad:As the situation remains tense in th...   
4  The blockade of key routes between Rawalpindi ...   

                                                Link Time Posted  \
0                  https://www.dawn.com/news/1863239  2 days ago 

In [ ]:
pip install ntscraper

In [ ]:
import time
import random
import pandas as pd
from ntscraper import Nitter

# Initialize the scraper
scraper = Nitter()

# Function to switch instances if needed
def switch_nitter_instance(scraper, instance_url):
    scraper.base_url = instance_url
    print(f"Switched to Nitter instance: {instance_url}")

# Function to randomize sleep
def random_sleep(min_time=1, max_time=3):
    sleep_time = random.uniform(min_time, max_time)
    time.sleep(sleep_time)

# List of Nitter instances to switch between if one fails
nitter_instances = [
    "https://nitter.net",
    "https://nitter.42l.fr",
    "https://nitter.snopyta.org"
]
major_roads = [
    # Islamabad/Rawalpindi
    "Murree Road", "9th Avenue", "IJP Road", "Airport Road", "Saddar Road", "Faizabad", "Kashmir Highway",
    "Margalla Road", "Jinnah Avenue", "Blue Area", "Peshawar Road", "GT Road", "Expressway", "Club Road",
    "Pir Wadhai Road", "Raja Bazaar", "Bahria Town Road",

    # Lahore
    "Ferozepur Road", "Canal Road", "MM Alam Road", "Mall Road", "Jail Road", "Lahore Ring Road",
    "Multan Road", "GT Road", "Defence Road", "Allama Iqbal Road", "College Road", "Raiwind Road",
    "Walton Road", "Bedian Road", "Shahrah-e-Quaid-e-Azam", "Wahdat Road",

    # Karachi
    "Shahrah-e-Faisal", "Rashid Minhas Road", "MA Jinnah Road", "Korangi Road", "University Road",
    "Lyari Expressway", "Karachi-Hyderabad Motorway (M9)", "Karsaz Road", "Sharah-e-Pakistan",
    "Nazimabad Road", "Saddar", "II Chundrigar Road", "Clifton Road", "Kemari Road",

    # Multan
    "Bosan Road", "Abdali Road", "Khanewal Road", "Vehari Road", "Sher Shah Road", "Multan Road",
    "Qadirpur Raan Road", "Old Shujabad Road", "Gulgasht Colony Road", "Chungi Road", "Nishter Road",

    # National Highways
    "Motorway M2", "Motorway M3", "Motorway M4", "GT Road", "N5 National Highway", "Indus Highway",
    "RCD Highway", "Makran Coastal Highway", "Karakoram Highway", "Silk Route",

    # Northern Areas / Tourist Destinations
    ## Swat
    "Mingora Road", "Saidu Sharif Road", "Malam Jabba Road", "Mingora-Fizagat Road",
    "Kalam Road", "Swat Expressway", "Bahrain-Kalam Road", "Matta Road", "Behrain Road", "Utror Road",
    ## Gilgit-Baltistan
    "Karakoram Highway", "Gilgit-Skardu Road", "Gilgit-Chilas Road", "Astore Road", "Shangrila Road", "Rama Road",
    "Deosai Road", "Gupis Road", "Phandar Road", "Naltar Road", "Ghizer Road", "Shigar Road",
    ## Hunza
    "Karakoram Highway", "Karimabad Road", "Passu Road", "Sost Road", "Attabad Tunnel Road",
    "Gulmit Road", "Aliabad Road", "Hussaini Suspension Bridge Road", "Borith Lake Road",
    ## Chitral
    "Chitral-Mastuj Road", "Lowari Tunnel Road", "Drosh Road", "Booni Road", "Shandur Pass Road",
    "Garum Chashma Road", "Kalash Valley Road", "Ayun Road",
    ## Murree and Galiyat
    "Murree Expressway", "Mall Road Murree", "Patriata Road", "New Murree Road", "Bhurban Road", "Ayubia Road",
    "Nathia Gali Road", "Dunga Gali Road", "Changla Gali Road", "Kashmir Point Road",
    ## Neelum Valley (Azad Kashmir)
    "Muzaffarabad-Neelum Road", "Keran Road", "Sharda-Kel Road", "Athmuqam Road", "Dudhnial Road",
    "Taobat Road", "Ratti Gali Lake Road", "Leepa Valley Road",
    ## Kaghan and Naran
    "Mansehra-Naran-Jalkhad Road", "Kaghan Valley Road", "Naran-Babusar Top Road", "Lulusar Lake Road",
    "Saiful Muluk Road", "Jalkhad-Chilas Road", "Shogran Road", "Balakot Road", "Kunhar River Road",
    ## Skardu
    "Skardu-Jaglot Road", "Skardu-Deosai Road", "Sadpara Road", "Khaplu Road", "Sheosar Lake Road",
    "Shigar Valley Road", "Shangrila Road", "Katpana Road", "Basho Valley Road",
    ## Other Northern Routes
    "Fairy Meadows Road", "Nanga Parbat Base Camp Road", "Rupal Valley Road", "Ratti Gali Lake Road",
    "Minimarg Road", "Khalti Lake Road"
]
english_keywords = [
    "accident", "collision", "roadblock", "container", "road safety", "road closure", "traffic",
    "blockage", "road hazard", "pile-up", "road accident", "traffic jam"
]

urdu_keywords = [
    "حادثہ", "ٹریفک", "سڑک بند", "سڑک حادثہ", "روڈ بلاک", "ٹریفک جام", "پل بند", "راستہ بند",
    "سڑک ٹوٹ گئی", "سیلاب", "رکاوٹ", "بارش", "پھسلن", "رکاوٹ", "ٹریفک پولیس", "ٹرک", "کنٹینر"
]

hashtags = ["#ChampAlertsOnTheGo", "#CAOTG"]

# Combine all keywords into one list
combined_keywords = major_roads + english_keywords + urdu_keywords + hashtags

# Dictionary to store tweet data
data = {
    'link': [],
    'text': [],
    'user': [],
    'likes': [],
    'retweets': [],
    'comments': []
}

# Log failed keywords for retry later
failed_keywords = []

# Start with the first Nitter instance
current_instance_index = 0
switch_nitter_instance(scraper, nitter_instances[current_instance_index])

# Set the number of tweets to retrieve per keyword
num_of_tweets = 50  # Adjust as necessary

# Geolocation filtering based on cities or regions in Pakistan
target_locations = [
    "Islamabad", "Rawalpindi", "Lahore", "Karachi", "Multan",
    "Peshawar", "Sialkot", "Faisalabad", "Quetta", "Sargodha"
]

# Function to filter tweets by location mentioned in bio or tweet
def geo_filter_tweet(tweet, target_locations):
    # Safely access the user's bio, use an empty string if the bio is missing
    user_bio = tweet['user'].get('bio', '').lower()

    # Safely access the tweet text
    tweet_text = tweet['text'].lower()

    # Check if any target location is mentioned in the bio or tweet text
    for location in target_locations:
        location_lower = location.lower()
        if location_lower in user_bio or location_lower in tweet_text:
            return True
    return False



# Function to avoid duplicates based on tweet URLs
def is_unique_tweet(tweet_link, crawled_links):
    if tweet_link not in crawled_links:
        crawled_links.add(tweet_link)
        return True
    return False

# Track unique tweet links to avoid duplicates
crawled_links = set()

# Loop through combined keywords to find tweets
for keyword in combined_keywords:
    try:
        tweets_data = scraper.get_tweets(keyword, mode='term', number=num_of_tweets)

        if tweets_data and 'tweets' in tweets_data:
            print(f"Found tweets for keyword: {keyword}")

            for tweet in tweets_data['tweets']:
                tweet_link = tweet['link']

                # Check if the tweet matches the geolocation criteria
                if geo_filter_tweet(tweet, target_locations):
                    if is_unique_tweet(tweet_link, crawled_links):  # Ensure no duplicates
                        data['link'].append(tweet_link)
                        data['text'].append(tweet['text'])
                        data['user'].append(tweet['user']['name'])
                        data['likes'].append(tweet['stats']['likes'])
                        data['retweets'].append(tweet['stats']['retweets'])
                        data['comments'].append(tweet['stats']['comments'])
                else:
                    print(f"Tweet did not match target location: {keyword}")
        else:
            print(f"No tweets found for keyword: {keyword}")

    except ValueError as e:
        print(f"Failed for keyword: {keyword} due to instance issues.")
        failed_keywords.append(keyword)

        # Switch to the next instance if available
        current_instance_index += 1
        if current_instance_index < len(nitter_instances):
            switch_nitter_instance(scraper, nitter_instances[current_instance_index])
        else:
            print("No more Nitter instances available. Stopping.")
            break
        random_sleep(5, 10)  # Sleep longer before retrying

    random_sleep()  # Add random sleep to avoid being blocked

# Check if any tweets were collected and save to CSV
if data['text']:
    df = pd.DataFrame(data)
    df.to_csv('road_safety_tweets_pakistan_r.csv', index=False)
    print("Tweets saved to road_safety_tweets_pakistan.csv")
else:
    print("No tweets found for any of the provided keywords.")

# Print failed keywords for retrying
if failed_keywords:
    print("Retry the following keywords due to instance issues:", failed_keywords)

Testing instances: 100%|██████████| 16/16 [00:05<00:00,  2.79it/s]
INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Switched to Nitter instance: https://nitter.net


Found tweets for keyword: Murree Road


INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Failed for keyword: 9th Avenue due to instance issues.
Switched to Nitter instance: https://nitter.42l.fr


...

Found tweets for keyword: کنٹینر


INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Found tweets for keyword: #ChampAlertsOnTheGo


INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Found tweets for keyword: #CAOTG
No tweets found for any of the provided keywords.
Retry the following keywords due to instance issues: ['9th Avenue']


In [ ]:
import time
import random
import pandas as pd
from ntscraper import Nitter

# Initialize the scraper
scraper = Nitter()

# Function to switch instances if needed
def switch_nitter_instance(scraper, instance_url):
    scraper.base_url = instance_url
    print(f"Switched to Nitter instance: {instance_url}")

# Function to randomize sleep
def random_sleep(min_time=1, max_time=3):
    sleep_time = random.uniform(min_time, max_time)
    time.sleep(sleep_time)

# List of Nitter instances to switch between if one fails
nitter_instances = [
    "https://nitter.net",
    "https://nitter.42l.fr",
    "https://nitter.snopyta.org",
    "https://nitter.privacydev.net"
]

# Dictionary to store tweet data
data = {
    'link': [],
    'text': [],
    'user': [],
    'likes': [],
    'retweets': [],
    'comments': []
}

# Log failed keywords for retry later
failed_keywords = []

# Start with the first Nitter instance
current_instance_index = 0
switch_nitter_instance(scraper, nitter_instances[current_instance_index])

# Set the number of tweets to retrieve per keyword
num_of_tweets = 50  # Adjust as necessary

# Function to avoid duplicates based on tweet URLs
def is_unique_tweet(tweet_link, crawled_links):
    if tweet_link not in crawled_links:
        crawled_links.add(tweet_link)
        return True
    return False

# Track unique tweet links to avoid duplicates
crawled_links = set()

# Function to filter tweets by location mentioned in bio or tweet
def geo_filter_tweet(tweet, target_locations):
    user_bio = tweet['user'].get('bio', '').lower()
    tweet_text = tweet['text'].lower()

    for location in target_locations:
        location_lower = location.lower()
        if location_lower in user_bio or location_lower in tweet_text:
            return True
    return False

major_roads = [
    # Islamabad/Rawalpindi
    "Murree Road", "9th Avenue", "IJP Road", "Airport Road", "Saddar Road", "Faizabad", "Kashmir Highway",
    "Margalla Road", "Jinnah Avenue", "Blue Area", "Peshawar Road", "GT Road", "Expressway", "Club Road",
    "Pir Wadhai Road", "Raja Bazaar", "Bahria Town Road",

    # Lahore
    "Ferozepur Road", "Canal Road", "MM Alam Road", "Mall Road", "Jail Road", "Lahore Ring Road",
    "Multan Road", "GT Road", "Defence Road", "Allama Iqbal Road", "College Road", "Raiwind Road",
    "Walton Road", "Bedian Road", "Shahrah-e-Quaid-e-Azam", "Wahdat Road",

    # Karachi
    "Shahrah-e-Faisal", "Rashid Minhas Road", "MA Jinnah Road", "Korangi Road", "University Road",
    "Lyari Expressway", "Karachi-Hyderabad Motorway (M9)", "Karsaz Road", "Sharah-e-Pakistan",
    "Nazimabad Road", "Saddar", "II Chundrigar Road", "Clifton Road", "Kemari Road",

    # Multan
    "Bosan Road", "Abdali Road", "Khanewal Road", "Vehari Road", "Sher Shah Road", "Multan Road",
    "Qadirpur Raan Road", "Old Shujabad Road", "Gulgasht Colony Road", "Chungi Road", "Nishter Road",

    # National Highways
    "Motorway M2", "Motorway M3", "Motorway M4", "GT Road", "N5 National Highway", "Indus Highway",
    "RCD Highway", "Makran Coastal Highway", "Karakoram Highway", "Silk Route",

    # Northern Areas / Tourist Destinations
    ## Swat
    "Mingora Road", "Saidu Sharif Road", "Malam Jabba Road", "Mingora-Fizagat Road",
    "Kalam Road", "Swat Expressway", "Bahrain-Kalam Road", "Matta Road", "Behrain Road", "Utror Road",
    ## Gilgit-Baltistan
    "Karakoram Highway", "Gilgit-Skardu Road", "Gilgit-Chilas Road", "Astore Road", "Shangrila Road", "Rama Road",
    "Deosai Road", "Gupis Road", "Phandar Road", "Naltar Road", "Ghizer Road", "Shigar Road",
    ## Hunza
    "Karakoram Highway", "Karimabad Road", "Passu Road", "Sost Road", "Attabad Tunnel Road",
    "Gulmit Road", "Aliabad Road", "Hussaini Suspension Bridge Road", "Borith Lake Road",
    ## Chitral
    "Chitral-Mastuj Road", "Lowari Tunnel Road", "Drosh Road", "Booni Road", "Shandur Pass Road",
    "Garum Chashma Road", "Kalash Valley Road", "Ayun Road",
    ## Murree and Galiyat
    "Murree Expressway", "Mall Road Murree", "Patriata Road", "New Murree Road", "Bhurban Road", "Ayubia Road",
    "Nathia Gali Road", "Dunga Gali Road", "Changla Gali Road", "Kashmir Point Road",
    ## Neelum Valley (Azad Kashmir)
    "Muzaffarabad-Neelum Road", "Keran Road", "Sharda-Kel Road", "Athmuqam Road", "Dudhnial Road",
    "Taobat Road", "Ratti Gali Lake Road", "Leepa Valley Road",
    ## Kaghan and Naran
    "Mansehra-Naran-Jalkhad Road", "Kaghan Valley Road", "Naran-Babusar Top Road", "Lulusar Lake Road",
    "Saiful Muluk Road", "Jalkhad-Chilas Road", "Shogran Road", "Balakot Road", "Kunhar River Road",
    ## Skardu
    "Skardu-Jaglot Road", "Skardu-Deosai Road", "Sadpara Road", "Khaplu Road", "Sheosar Lake Road",
    "Shigar Valley Road", "Shangrila Road", "Katpana Road", "Basho Valley Road",
    ## Other Northern Routes
    "Fairy Meadows Road", "Nanga Parbat Base Camp Road", "Rupal Valley Road", "Ratti Gali Lake Road",
    "Minimarg Road", "Khalti Lake Road"
]
english_keywords = [
    "accident", "collision", "roadblock", "container", "road safety", "road closure", "traffic",
    "blockage", "road hazard", "pile-up", "road accident", "traffic jam"
]

urdu_keywords = [
    "حادثہ", "ٹریفک", "سڑک بند", "سڑک حادثہ", "روڈ بلاک", "ٹریفک جام", "پل بند", "راستہ بند",
    "سڑک ٹوٹ گئی", "سیلاب", "رکاوٹ", "بارش", "پھسلن", "رکاوٹ", "ٹریفک پولیس", "ٹرک", "کنٹینر"
]

hashtags = ["#ChampAlertsOnTheGo", "#CAOTG"]

# Combine all keywords into one list
combined_keywords = major_roads + english_keywords + urdu_keywords + hashtags
# Define a retry limit
retry_limit = 3
retry_counter = 0

# Loop through combined keywords to find tweets
for keyword in combined_keywords:
    retry_counter = 0  # Reset retry counter for each keyword

    while retry_counter < retry_limit:
        try:
            tweets_data = scraper.get_tweets(keyword, mode='term', number=num_of_tweets)

            if tweets_data and 'tweets' in tweets_data:
                print(f"Found tweets for keyword: {keyword}")

                for tweet in tweets_data['tweets']:
                    tweet_link = tweet['link']

                    # Check if the tweet matches the geolocation criteria
                    if geo_filter_tweet(tweet, target_locations):
                        if is_unique_tweet(tweet_link, crawled_links):  # Ensure no duplicates
                            data['link'].append(tweet_link)
                            data['text'].append(tweet['text'])
                            data['user'].append(tweet['user']['name'])
                            data['likes'].append(tweet['stats']['likes'])
                            data['retweets'].append(tweet['stats']['retweets'])
                            data['comments'].append(tweet['stats']['comments'])
                    else:
                        print(f"Tweet did not match target location: {keyword}")
                break  # Successfully retrieved tweets, break out of retry loop
            else:
                print(f"No tweets found for keyword: {keyword}")
                break  # No tweets, move to the next keyword

        except Exception as e:
            print(f"Failed for keyword: {keyword} due to {e}. Retrying...")

            # Increment the retry counter
            retry_counter += 1
            if retry_counter >= retry_limit:
                failed_keywords.append(keyword)
                print(f"Failed for keyword: {keyword} after {retry_limit} attempts.")

            # Switch to the next instance if available
            current_instance_index += 1
            if current_instance_index < len(nitter_instances):
                switch_nitter_instance(scraper, nitter_instances[current_instance_index])
            else:
                print("No more Nitter instances available. Stopping.")
                break

            random_sleep(5, 10)  # Sleep longer before retrying

    random_sleep()  # Add random sleep to avoid being blocked

# Check if any tweets were collected and save to CSV
if data['text']:
    df = pd.DataFrame(data)
    df.to_csv('road_safety_tweets_pakistan.csv', index=False)
    print("Tweets saved to road_safety_tweets_pakistan.csv")
else:
    print("No tweets found for any of the provided keywords.")

# Print failed keywords for retrying
if failed_keywords:
    print("Retry the following keywords due to instance issues:", failed_keywords)


SyntaxError: invalid syntax. Perhaps you forgot a comma? (<ipython-input-8-d83a6b01e7ef>, line 75)

In [ ]:
import time
import random
import pandas as pd
from ntscraper import Nitter

# Initialize the scraper
scraper = Nitter()

# Function to switch instances if needed
def switch_nitter_instance(scraper, instance_url):
    scraper.base_url = instance_url
    print(f"Switched to Nitter instance: {instance_url}")

# Function to randomize sleep
def random_sleep(min_time=1, max_time=3):
    sleep_time = random.uniform(min_time, max_time)
    time.sleep(sleep_time)

# List of Nitter instances to switch between if one fails
nitter_instances = [
    "https://nitter.net",
    "https://nitter.42l.fr",
    "https://nitter.snopyta.org",
    "https://nitter.privacydev.net"
]

# Dictionary to store tweet data
data = {
    'link': [],
    'text': [],
    'user': [],
    'likes': [],
    'retweets': [],
    'comments': []
}

# Function to save tweets to a CSV immediately after tweets are found
def save_to_csv(data, filename='road_safety_tweets_pakistan.csv'):
    df = pd.DataFrame(data)
    df.to_csv(filename, mode='a', header=not pd.io.common.file_exists(filename), index=False)

# Log failed keywords for retry later
failed_keywords = []
remaining_keywords = []

# Start with the first Nitter instance
current_instance_index = 0
switch_nitter_instance(scraper, nitter_instances[current_instance_index])

# Set the number of tweets to retrieve per keyword
num_of_tweets = 50  # Adjust as necessary

# Function to avoid duplicates based on tweet URLs
def is_unique_tweet(tweet_link, crawled_links):
    if tweet_link not in crawled_links:
        crawled_links.add(tweet_link)
        return True
    return False

# Track unique tweet links to avoid duplicates
crawled_links = set()

# Function to filter tweets by location mentioned in bio or tweet
def geo_filter_tweet(tweet, target_locations):
    user_bio = tweet['user'].get('bio', '').lower()
    tweet_text = tweet['text'].lower()

    for location in target_locations:
        location_lower = location.lower()
        if location_lower in user_bio or location_lower in tweet_text:
            return True
    return False

# Keywords and locations
major_roads = [
    # Islamabad/Rawalpindi
    "Murree Road", "9th Avenue", "IJP Road", "Airport Road", "Saddar Road", "Faizabad", "Kashmir Highway",
    "Margalla Road", "Jinnah Avenue", "Blue Area", "Peshawar Road", "GT Road", "Expressway", "Club Road",
    "Pir Wadhai Road", "Raja Bazaar", "Bahria Town Road",

    # Lahore
    "Ferozepur Road", "Canal Road", "MM Alam Road", "Mall Road", "Jail Road", "Lahore Ring Road",
    "Multan Road", "GT Road", "Defence Road", "Allama Iqbal Road", "College Road", "Raiwind Road",
    "Walton Road", "Bedian Road", "Shahrah-e-Quaid-e-Azam", "Wahdat Road",

    # Karachi
    "Shahrah-e-Faisal", "Rashid Minhas Road", "MA Jinnah Road", "Korangi Road", "University Road",
    "Lyari Expressway", "Karachi-Hyderabad Motorway (M9)", "Karsaz Road", "Sharah-e-Pakistan",
    "Nazimabad Road", "Saddar", "II Chundrigar Road", "Clifton Road", "Kemari Road",

    # Multan
    "Bosan Road", "Abdali Road", "Khanewal Road", "Vehari Road", "Sher Shah Road", "Multan Road",
    "Qadirpur Raan Road", "Old Shujabad Road", "Gulgasht Colony Road", "Chungi Road", "Nishter Road",

    # National Highways
    "Motorway M2", "Motorway M3", "Motorway M4", "GT Road", "N5 National Highway", "Indus Highway",
    "RCD Highway", "Makran Coastal Highway", "Karakoram Highway", "Silk Route",

    # Northern Areas / Tourist Destinations
    ## Swat
    "Mingora Road", "Saidu Sharif Road", "Malam Jabba Road", "Mingora-Fizagat Road",
    "Kalam Road", "Swat Expressway", "Bahrain-Kalam Road", "Matta Road", "Behrain Road", "Utror Road",
    ## Gilgit-Baltistan
    "Karakoram Highway", "Gilgit-Skardu Road", "Gilgit-Chilas Road", "Astore Road", "Shangrila Road", "Rama Road",
    "Deosai Road", "Gupis Road", "Phandar Road", "Naltar Road", "Ghizer Road", "Shigar Road",
    ## Hunza
    "Karakoram Highway", "Karimabad Road", "Passu Road", "Sost Road", "Attabad Tunnel Road",
    "Gulmit Road", "Aliabad Road", "Hussaini Suspension Bridge Road", "Borith Lake Road",
    ## Chitral
    "Chitral-Mastuj Road", "Lowari Tunnel Road", "Drosh Road", "Booni Road", "Shandur Pass Road",
    "Garum Chashma Road", "Kalash Valley Road", "Ayun Road",
    ## Murree and Galiyat
    "Murree Expressway", "Mall Road Murree", "Patriata Road", "New Murree Road", "Bhurban Road", "Ayubia Road",
    "Nathia Gali Road", "Dunga Gali Road", "Changla Gali Road", "Kashmir Point Road",
    ## Neelum Valley (Azad Kashmir)
    "Muzaffarabad-Neelum Road", "Keran Road", "Sharda-Kel Road", "Athmuqam Road", "Dudhnial Road",
    "Taobat Road", "Ratti Gali Lake Road", "Leepa Valley Road",
    ## Kaghan and Naran
    "Mansehra-Naran-Jalkhad Road", "Kaghan Valley Road", "Naran-Babusar Top Road", "Lulusar Lake Road",
    "Saiful Muluk Road", "Jalkhad-Chilas Road", "Shogran Road", "Balakot Road", "Kunhar River Road",
    ## Skardu
    "Skardu-Jaglot Road", "Skardu-Deosai Road", "Sadpara Road", "Khaplu Road", "Sheosar Lake Road",
    "Shigar Valley Road", "Shangrila Road", "Katpana Road", "Basho Valley Road",
    ## Other Northern Routes
    "Fairy Meadows Road", "Nanga Parbat Base Camp Road", "Rupal Valley Road", "Ratti Gali Lake Road",
    "Minimarg Road", "Khalti Lake Road"
]
english_keywords = [
    "accident", "collision", "roadblock", "container", "road safety", "road closure", "traffic",
    "blockage", "road hazard", "pile-up", "road accident", "traffic jam"
]

urdu_keywords = [
    "حادثہ", "ٹریفک", "سڑک بند", "سڑک حادثہ", "روڈ بلاک", "ٹریفک جام", "پل بند", "راستہ بند",
    "سڑک ٹوٹ گئی", "سیلاب", "رکاوٹ", "بارش", "پھسلن", "رکاوٹ", "ٹریفک پولیس", "ٹرک", "کنٹینر"
]

hashtags = ["#ChampAlertsOnTheGo", "#CAOTG"]

# Combine all keywords into one list
combined_keywords = major_roads + english_keywords + urdu_keywords + hashtags

# Define a retry limit
retry_limit = 3
retry_counter = 0

# Loop through combined keywords to find tweets
for keyword in combined_keywords:
    retry_counter = 0  # Reset retry counter for each keyword
    keyword_data = {  # Temporary storage for data of this keyword
        'link': [],
        'text': [],
        'user': [],
        'likes': [],
        'retweets': [],
        'comments': []
    }

    while retry_counter < retry_limit:
        try:
            tweets_data = scraper.get_tweets(keyword, mode='term', number=num_of_tweets)

            if tweets_data and 'tweets' in tweets_data:
                print(f"Found tweets for keyword: {keyword}")

                for tweet in tweets_data['tweets']:
                    tweet_link = tweet['link']

                    # Check if the tweet matches the geolocation criteria
                    if geo_filter_tweet(tweet, target_locations):
                        if is_unique_tweet(tweet_link, crawled_links):  # Ensure no duplicates
                            keyword_data['link'].append(tweet_link)
                            keyword_data['text'].append(tweet['text'])
                            keyword_data['user'].append(tweet['user']['name'])
                            keyword_data['likes'].append(tweet['stats']['likes'])
                            keyword_data['retweets'].append(tweet['stats']['retweets'])
                            keyword_data['comments'].append(tweet['stats']['comments'])
                    else:
                        print(f"Tweet did not match target location: {keyword}")

                # Save tweets found for this keyword immediately
                if keyword_data['text']:
                    save_to_csv(keyword_data)
                    print(f"Tweets for keyword '{keyword}' saved to CSV.")

                break  # Successfully retrieved tweets, break out of retry loop
            else:
                print(f"No tweets found for keyword: {keyword}")
                break  # No tweets, move to the next keyword

        except Exception as e:
            print(f"Failed for keyword: {keyword} due to {e}. Retrying...")

            # Increment the retry counter
            retry_counter += 1
            if retry_counter >= retry_limit:
                failed_keywords.append(keyword)
                print(f"Failed for keyword: {keyword} after {retry_limit} attempts.")
                break

            # Switch to the next instance if available
            if retry_counter == 1:  # Switch after the first failure to avoid rate limiting
                current_instance_index += 1
                if current_instance_index < len(nitter_instances):
                    switch_nitter_instance(scraper, nitter_instances[current_instance_index])
                else:
                    print("No more Nitter instances available. Stopping.")
                    remaining_keywords.append(keyword)
                    break

            random_sleep(5, 10)  # Sleep longer before retrying

    random_sleep()  # Add random sleep to avoid being blocked

# If there are keywords left to retry later, print them
if failed_keywords or remaining_keywords:
    print(f"Failed keywords due to instance issues: {failed_keywords}")
    print(f"Remaining keywords that were not processed: {remaining_keywords}")
else:
    print("All keywords processed successfully.")


ModuleNotFoundError: No module named 'ntscraper'

In [ ]:
import time
import random
import pandas as pd
from datetime import datetime, timedelta
from ntscraper import Nitter

# Initialize the scraper
scraper = Nitter()

def switch_nitter_instance(scraper, instance_url):
    scraper.base_url = instance_url
    print(f"Switched to Nitter instance: {instance_url}")

def random_sleep(min_time=1, max_time=3):
    sleep_time = random.uniform(min_time, max_time)
    time.sleep(sleep_time)

nitter_instances = [
    "https://nitter.net",
    "https://nitter.42l.fr",
    "https://nitter.snopyta.org",
    "https://nitter.privacydev.net"
]

data = {
    'link': [],
    'text': [],
    'user': [],
    'likes': [],
    'retweets': [],
    'comments': [],
    'timestamp': []
}

def save_to_csv(data, filename='road_safety_tweets_pakistan.csv'):
    df = pd.DataFrame(data)
    if not df.empty:
        print(f"Saving {len(df)} rows to CSV: {filename}")
        df.to_csv(filename, mode='a', header=not pd.io.common.file_exists(filename), index=False)

def filter_recent_tweets(tweets, minutes=15):
    recent_tweets = []
    current_time = datetime.now()
    time_limit = current_time - timedelta(minutes=minutes)

    for tweet in tweets:
        tweet_time = datetime.strptime(tweet['timestamp'], "%Y-%m-%d %H:%M:%S")
        if tweet_time >= time_limit:
            recent_tweets.append(tweet)
    return recent_tweets

def is_unique_tweet(tweet_link, crawled_links):
    if tweet_link not in crawled_links:
        crawled_links.add(tweet_link)
        return True
    return False

crawled_links = set()
current_instance_index = 0
switch_nitter_instance(scraper, nitter_instances[current_instance_index])

major_roads = ["Murree Road", "Canal Road"]  # Shortened for brevity
english_keywords = ["accident", "collision"]
urdu_keywords = ["حادثہ", "ٹریفک"]
hashtags = ["#ChampAlertsOnTheGo", "#CAOTG"]

combined_keywords = major_roads + english_keywords + urdu_keywords + hashtags

for keyword in combined_keywords:
    try:
        print(f"Fetching tweets for keyword: {keyword}")
        tweets_data = scraper.get_tweets(keyword, mode='term', number=100)

        # Debugging: Print raw tweets data
        print(f"Raw data for keyword '{keyword}': {tweets_data}")

        if tweets_data and 'tweets' in tweets_data:
            recent_tweets = filter_recent_tweets(tweets_data['tweets'], minutes=15)
            print(f"Filtered {len(recent_tweets)} recent tweets for keyword: {keyword}")

            for tweet in recent_tweets:
                tweet_link = tweet['link']
                if is_unique_tweet(tweet_link, crawled_links):
                    data['link'].append(tweet_link)
                    data['text'].append(tweet['text'])
                    data['user'].append(tweet['user']['name'])
                    data['likes'].append(tweet['stats']['likes'])
                    data['retweets'].append(tweet['stats']['retweets'])
                    data['comments'].append(tweet['stats']['comments'])
                    data['timestamp'].append(tweet['timestamp'])

            save_to_csv(data)
        else:
            print(f"No tweets found or invalid data for keyword: {keyword}")

    except Exception as e:
        print(f"Error while processing keyword '{keyword}': {e}")

    random_sleep()

print(f"Final data: {data}")
if data['link']:
    save_to_csv(data)
else:
    print("No data to save.")


ModuleNotFoundError: No module named 'ntscraper'

In [ ]:
pip install ntscraper

In [ ]:
import time
import random
import pandas as pd
from ntscraper import Nitter

# Initialize the scraper
scraper = Nitter()

# Function to switch instances if needed
def switch_nitter_instance(scraper, instance_url):
    scraper.base_url = instance_url
    print(f"Switched to Nitter instance: {instance_url}")

# Function to randomize sleep
def random_sleep(min_time=1, max_time=3):
    sleep_time = random.uniform(min_time, max_time)
    time.sleep(sleep_time)

# List of Nitter instances to switch between if one fails
nitter_instances = [
    "https://nitter.net",
    "https://nitter.42l.fr",
    "https://nitter.snopyta.org",
    "https://nitter.privacydev.net",
    "https://lightbrd.com"
]

# List of major roads in key Pakistani cities (Islamabad, Lahore, Karachi, Multan, etc.)
major_roads = [
    "Murree Road", "9th Avenue", "IJP Road", "Airport Road", "Saddar Road", "Faizabad", "Kashmir Highway",
    "Margalla Road", "Jinnah Avenue", "Blue Area", "Peshawar Road", "GT Road", "Expressway", "Club Road"
]
# Example hazardous keywords
english_keywords = ["accident", "collision", "roadblock", "traffic"]
urdu_keywords = ["حادثہ", "ٹریفک", "سڑک بند", "روڈ بلاک"]

# Combine all keywords into one list
combined_keywords = english_keywords + urdu_keywords

# Create combinations
queries = [f"{road} {keyword}" for road in major_roads for keyword in combined_keywords]

# Dictionary to store tweet data
data = {
    'road': [],
    'hazardous_keyword': [],
    'link': [],
    'text': [],
    'user': [],
    'likes': [],
    'retweets': [],
    'comments': []
}

# Log failed queries for retry later
failed_queries = []

# Start with the first Nitter instance
current_instance_index = 0
switch_nitter_instance(scraper, nitter_instances[current_instance_index])

# Function to avoid duplicates based on tweet URLs
def is_unique_tweet(tweet_link, crawled_links):
    if tweet_link not in crawled_links:
        crawled_links.add(tweet_link)
        return True
    return False

# Track unique tweet links to avoid duplicates
crawled_links = set()

# Set the number of tweets to retrieve per query
num_of_tweets = 50  # Adjust as necessary

# Loop through all queries
for query in queries:
    road, keyword = query.split(" ", 1)  # Extract road and keyword from query
    try:
        tweets_data = scraper.get_tweets(query, mode='term', number=num_of_tweets)

        if tweets_data and 'tweets' in tweets_data:
            print(f"Found tweets for query: {query}")

            for tweet in tweets_data['tweets']:
                tweet_link = tweet['link']

                if is_unique_tweet(tweet_link, crawled_links):  # Ensure no duplicates
                    data['road'].append(road)
                    data['hazardous_keyword'].append(keyword)
                    data['link'].append(tweet_link)
                    data['text'].append(tweet['text'])
                    data['user'].append(tweet['user']['name'])
                    data['likes'].append(tweet['stats']['likes'])
                    data['retweets'].append(tweet['stats']['retweets'])
                    data['comments'].append(tweet['stats']['comments'])

            # Save data incrementally after each successful query
            df = pd.DataFrame(data)
            df.to_csv('road_safety_combination_tweets.csv', index=False)
            print(f"Tweets for query '{query}' saved successfully.")
        else:
            print(f"No tweets found for query: {query}")

    except Exception as e:
        print(f"Failed for query: {query} due to {e}")
        failed_queries.append(query)

        # Switch to the next instance if available
        current_instance_index += 1
        if current_instance_index < len(nitter_instances):
            switch_nitter_instance(scraper, nitter_instances[current_instance_index])
        else:
            print("No more Nitter instances available. Stopping.")
            break
        random_sleep(5, 10)  # Sleep longer before retrying

    random_sleep()  # Add random sleep to avoid being blocked

# Final saving and summary
if data['text']:
    df = pd.DataFrame(data)
    df.to_csv('road_safety_combination_tweets.csv', index=False)
    print("Final tweets saved to road_safety_combination_tweets.csv.")
else:
    print("No tweets were collected.")

if failed_queries:
    print("Retry the following queries due to issues:", failed_queries)
else:
    print("All queries processed successfully.")


Testing instances: 100%|██████████| 10/10 [00:14<00:00,  1.45s/it]
INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Switched to Nitter instance: https://nitter.net


INFO:root:Current stats for Murree Road accident: 19 tweets, 0 threads...
INFO:root:Current stats for Murree Road accident: 31 tweets, 0 threads...


Found tweets for query: Murree Road accident
Tweets for query 'Murree Road accident' saved successfully.


INFO:root:No instance specified, using random instance https://nitter.privacydev.net
INFO:root:Current stats for Murree Road collision: 10 tweets, 0 threads...


...

Found tweets for query: Club Road ٹریفک
Tweets for query 'Club Road ٹریفک' saved successfully.


INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Found tweets for query: Club Road سڑک بند
Tweets for query 'Club Road سڑک بند' saved successfully.


INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Found tweets for query: Club Road روڈ بلاک
Tweets for query 'Club Road روڈ بلاک' saved successfully.
Final tweets saved to road_safety_combination_tweets.csv.
All queries processed successfully.


In [ ]:
import time
import random
import pandas as pd
from ntscraper import Nitter

# Initialize the scraper
scraper = Nitter()

# Function to switch instances if needed
def switch_nitter_instance(scraper, instance_url):
    scraper.base_url = instance_url
    print(f"Switched to Nitter instance: {instance_url}")

# Function to randomize sleep
def random_sleep(min_time=1, max_time=3):
    sleep_time = random.uniform(min_time, max_time)
    time.sleep(sleep_time)

# List of Nitter instances to switch between if one fails
nitter_instances = [
    "https://nitter.net",
    "https://nitter.42l.fr",
    "https://nitter.snopyta.org",
    "https://nitter.privacydev.net",
    "https://lightbrd.com"
]

# List of major roads in key Pakistani cities (Islamabad, Lahore, Karachi, Multan, etc.)
major_roads = [
    # Islamabad/Rawalpindi
    "Murree Road", "9th Avenue", "IJP Road", "Airport Road", "Saddar Road", "Faizabad", "Kashmir Highway",
    "Margalla Road", "Jinnah Avenue", "Blue Area", "Peshawar Road", "GT Road", "Expressway", "Club Road",
    "Pir Wadhai Road", "Raja Bazaar", "Bahria Town Road",

    # Lahore
    "Ferozepur Road", "Canal Road", "MM Alam Road", "Mall Road", "Jail Road", "Lahore Ring Road",
    "Multan Road", "GT Road", "Defence Road", "Allama Iqbal Road", "College Road", "Raiwind Road",
    "Walton Road", "Bedian Road", "Shahrah-e-Quaid-e-Azam", "Wahdat Road",

    # Karachi
    "Shahrah-e-Faisal", "Rashid Minhas Road", "MA Jinnah Road", "Korangi Road", "University Road",
    "Lyari Expressway", "Karachi-Hyderabad Motorway (M9)", "Karsaz Road", "Sharah-e-Pakistan",
    "Nazimabad Road", "Saddar", "II Chundrigar Road", "Clifton Road", "Kemari Road",

    # Multan
    "Bosan Road", "Abdali Road", "Khanewal Road", "Vehari Road", "Sher Shah Road", "Multan Road",
    "Qadirpur Raan Road", "Old Shujabad Road", "Gulgasht Colony Road", "Chungi Road", "Nishter Road",

    # National Highways
    "Motorway M2", "Motorway M3", "Motorway M4", "GT Road", "N5 National Highway", "Indus Highway",
    "RCD Highway", "Makran Coastal Highway", "Karakoram Highway", "Silk Route",

    # Northern Areas / Tourist Destinations
    ## Swat
    "Mingora Road", "Saidu Sharif Road", "Malam Jabba Road", "Mingora-Fizagat Road",
    "Kalam Road", "Swat Expressway", "Bahrain-Kalam Road", "Matta Road", "Behrain Road", "Utror Road",
    ## Gilgit-Baltistan
    "Karakoram Highway", "Gilgit-Skardu Road", "Gilgit-Chilas Road", "Astore Road", "Shangrila Road", "Rama Road",
    "Deosai Road", "Gupis Road", "Phandar Road", "Naltar Road", "Ghizer Road", "Shigar Road",
    ## Hunza
    "Karakoram Highway", "Karimabad Road", "Passu Road", "Sost Road", "Attabad Tunnel Road",
    "Gulmit Road", "Aliabad Road", "Hussaini Suspension Bridge Road", "Borith Lake Road",
    ## Chitral
    "Chitral-Mastuj Road", "Lowari Tunnel Road", "Drosh Road", "Booni Road", "Shandur Pass Road",
    "Garum Chashma Road", "Kalash Valley Road", "Ayun Road",
    ## Murree and Galiyat
    "Murree Expressway", "Mall Road Murree", "Patriata Road", "New Murree Road", "Bhurban Road", "Ayubia Road",
    "Nathia Gali Road", "Dunga Gali Road", "Changla Gali Road", "Kashmir Point Road",
    ## Neelum Valley (Azad Kashmir)
    "Muzaffarabad-Neelum Road", "Keran Road", "Sharda-Kel Road", "Athmuqam Road", "Dudhnial Road",
    "Taobat Road", "Ratti Gali Lake Road", "Leepa Valley Road",
    ## Kaghan and Naran
    "Mansehra-Naran-Jalkhad Road", "Kaghan Valley Road", "Naran-Babusar Top Road", "Lulusar Lake Road",
    "Saiful Muluk Road", "Jalkhad-Chilas Road", "Shogran Road", "Balakot Road", "Kunhar River Road",
    ## Skardu
    "Skardu-Jaglot Road", "Skardu-Deosai Road", "Sadpara Road", "Khaplu Road", "Sheosar Lake Road",
    "Shigar Valley Road", "Shangrila Road", "Katpana Road", "Basho Valley Road",
    ## Other Northern Routes
    "Fairy Meadows Road", "Nanga Parbat Base Camp Road", "Rupal Valley Road", "Ratti Gali Lake Road",
    "Minimarg Road", "Khalti Lake Road"
]
# Example hazardous keywords
english_keywords = [
    "accident", "collision", "roadblock", "container", "road safety", "road closure", "traffic",
    "blockage", "road hazard", "pile-up", "road accident", "traffic jam"
]

urdu_keywords = [
    "حادثہ", "ٹریفک", "سڑک بند", "سڑک حادثہ", "روڈ بلاک", "ٹریفک جام", "پل بند", "راستہ بند",
    "سڑک ٹوٹ گئی", "سیلاب", "رکاوٹ", "بارش", "پھسلن", "رکاوٹ", "ٹریفک پولیس", "ٹرک", "کنٹینر"
]

# Combine all keywords into one list
combined_keywords = english_keywords + urdu_keywords

# Create combinations
queries = [f"{road} {keyword}" for road in major_roads for keyword in combined_keywords]

# Dictionary to store tweet data
data = {
    'road': [],
    'hazardous_keyword': [],
    'link': [],
    'text': [],
    'user': [],
    'likes': [],
    'retweets': [],
    'comments': []
}

# Log failed queries for retry later
failed_queries = []

# Start with the first Nitter instance
current_instance_index = 0
switch_nitter_instance(scraper, nitter_instances[current_instance_index])

# Function to avoid duplicates based on tweet URLs
def is_unique_tweet(tweet_link, crawled_links):
    if tweet_link not in crawled_links:
        crawled_links.add(tweet_link)
        return True
    return False

# Track unique tweet links to avoid duplicates
crawled_links = set()

# Set the number of tweets to retrieve per query
num_of_tweets = 50  # Adjust as necessary

# Loop through all queries
for query in queries:
    road, keyword = query.split(" ", 1)  # Extract road and keyword from query
    try:
        tweets_data = scraper.get_tweets(query, mode='term', number=num_of_tweets)

        if tweets_data and 'tweets' in tweets_data:
            print(f"Found tweets for query: {query}")

            for tweet in tweets_data['tweets']:
                tweet_link = tweet['link']

                if is_unique_tweet(tweet_link, crawled_links):  # Ensure no duplicates
                    data['road'].append(road)
                    data['hazardous_keyword'].append(keyword)
                    data['link'].append(tweet_link)
                    data['text'].append(tweet['text'])
                    data['user'].append(tweet['user']['name'])
                    data['likes'].append(tweet['stats']['likes'])
                    data['retweets'].append(tweet['stats']['retweets'])
                    data['comments'].append(tweet['stats']['comments'])

            # Save data incrementally after each successful query
            df = pd.DataFrame(data)
            df.to_csv('road_safety_combination_tweets.csv', index=False)
            print(f"Tweets for query '{query}' saved successfully.")
        else:
            print(f"No tweets found for query: {query}")

    except Exception as e:
        print(f"Failed for query: {query} due to {e}")
        failed_queries.append(query)

        # Switch to the next instance if available
        current_instance_index += 1
        if current_instance_index < len(nitter_instances):
            switch_nitter_instance(scraper, nitter_instances[current_instance_index])
        else:
            print("No more Nitter instances available. Stopping.")
            break
        random_sleep(5, 10)  # Sleep longer before retrying

    random_sleep()  # Add random sleep to avoid being blocked

# Final saving and summary
if data['text']:
    df = pd.DataFrame(data)
    df.to_csv('road_safety_combination_tweets1.csv', index=False)
    print("Final tweets saved to road_safety_combination_tweets.csv.")
else:
    print("No tweets were collected.")

if failed_queries:
    print("Retry the following queries due to issues:", failed_queries)
else:
    print("All queries processed successfully.")


Testing instances: 100%|██████████| 10/10 [00:15<00:00,  1.54s/it]
INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Switched to Nitter instance: https://nitter.net


INFO:root:Current stats for Murree Road accident: 19 tweets, 0 threads...
INFO:root:Current stats for Murree Road accident: 31 tweets, 0 threads...


Found tweets for query: Murree Road accident
Tweets for query 'Murree Road accident' saved successfully.


INFO:root:No instance specified, using random instance https://nitter.privacydev.net
INFO:root:Current stats for Murree Road collision: 10 tweets, 0 threads...


...

Found tweets for query: Murree Road ٹریفک پولیس
Tweets for query 'Murree Road ٹریفک پولیس' saved successfully.


INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Found tweets for query: Murree Road ٹرک
Tweets for query 'Murree Road ٹرک' saved successfully.


INFO:root:No instance specified, using random instance https://lightbrd.com


Failed for query: Murree Road کنٹینر due to list index out of range
No more Nitter instances available. Stopping.
Final tweets saved to road_safety_combination_tweets.csv.
Retry the following queries due to issues: ['Murree Road roadblock', 'Murree Road road safety', 'Murree Road traffic jam', 'Murree Road سیلاب', 'Murree Road کنٹینر']


In [ ]:
import time
import random
import pandas as pd
from ntscraper import Nitter

# Initialize the scraper
scraper = Nitter()

# Function to switch instances if needed
def switch_nitter_instance(scraper, instance_url):
    scraper.base_url = instance_url
    print(f"Switched to Nitter instance: {instance_url}")

# Function to randomize sleep
def random_sleep(min_time=3, max_time=7):
    sleep_time = random.uniform(min_time, max_time)
    time.sleep(sleep_time)

# List of Nitter instances to switch between if one fails
nitter_instances = [
    "https://nitter.net",
    "https://nitter.42l.fr",
    "https://nitter.snopyta.org",
    "https://nitter.privacydev.net",
    "https://lightbrd.com"
]

# Keywords
# Define major roads
major_roads = [
    # Lahore
    "Ferozepur Road", "Canal Road", "MM Alam Road", "Mall Road", "Jail Road", "Lahore Ring Road",
    "Multan Road", "GT Road", "Defence Road", "Allama Iqbal Road", "College Road", "Raiwind Road",
    "Walton Road", "Bedian Road", "Shahrah-e-Quaid-e-Azam", "Wahdat Road",
    # Karachi
    "Shahrah-e-Faisal", "Rashid Minhas Road", "MA Jinnah Road", "Korangi Road", "University Road",
    "Lyari Expressway", "Karachi-Hyderabad Motorway (M9)", "Karsaz Road", "Sharah-e-Pakistan",
    "Nazimabad Road", "Saddar", "II Chundrigar Road", "Clifton Road", "Kemari Road",
    # Multan
    "Bosan Road", "Abdali Road", "Khanewal Road", "Vehari Road", "Sher Shah Road", "Multan Road",
    "Qadirpur Raan Road", "Old Shujabad Road", "Gulgasht Colony Road", "Chungi Road", "Nishter Road",
    # National Highways
    "Motorway M2", "Motorway M3", "Motorway M4", "GT Road", "N5 National Highway", "Indus Highway",
    "RCD Highway", "Makran Coastal Highway", "Karakoram Highway", "Silk Route"
]

# Hazard-related keywords
english_keywords = [
    "accident", "collision", "roadblock", "container", "road safety", "road closure", "traffic",
    "blockage", "road hazard", "pile-up", "road accident", "traffic jam"
]

urdu_keywords = [
    "حادثہ", "ٹریفک", "سڑک بند", "سڑک حادثہ", "روڈ بلاک", "ٹریفک جام", "پل بند", "راستہ بند",
    "سڑک ٹوٹ گئی", "سیلاب", "رکاوٹ", "بارش", "پھسلن", "رکاوٹ", "ٹریفک پولیس", "ٹرک", "کنٹینر"
]

hashtags = ["#ChampAlertsOnTheGo", "#CAOTG"]

# Combine all keywords
combined_keywords = english_keywords + urdu_keywords + hashtags
# Create combinations
combined_keywords = english_keywords + urdu_keywords
queries = [f"{road} {keyword}" for road in major_roads for keyword in combined_keywords]

# Dictionary to store tweet data
data = {
    'road': [],
    'hazardous_keyword': [],
    'link': [],
    'text': [],
    'user': [],
    'likes': [],
    'retweets': [],
    'comments': []
}

# Log failed queries for retry later
failed_queries = []

# Start with the first Nitter instance
current_instance_index = 0
switch_nitter_instance(scraper, nitter_instances[current_instance_index])

# Function to avoid duplicates based on tweet URLs
def is_unique_tweet(tweet_link, crawled_links):
    if tweet_link not in crawled_links:
        crawled_links.add(tweet_link)
        return True
    return False

# Track unique tweet links to avoid duplicates
crawled_links = set()

# Set the number of tweets to retrieve per query
num_of_tweets = 50  # Adjust as necessary

# Loop through all queries
for query in queries:
    road, keyword = query.split(" ", 1)  # Extract road and keyword from query
    instance_failed = False

    for instance in nitter_instances:
        try:
            switch_nitter_instance(scraper, instance)
            tweets_data = scraper.get_tweets(query, mode='term', number=num_of_tweets)

            if tweets_data and 'tweets' in tweets_data:
                print(f"Found tweets for query: {query}")

                for tweet in tweets_data['tweets']:
                    tweet_link = tweet.get('link', '')
                    tweet_text = tweet.get('text', '')

                    if is_unique_tweet(tweet_link, crawled_links):
                        data['road'].append(road)
                        data['hazardous_keyword'].append(keyword)
                        data['link'].append(tweet_link)
                        data['text'].append(tweet_text)
                        data['user'].append(tweet['user']['name'])
                        data['likes'].append(tweet['stats']['likes'])
                        data['retweets'].append(tweet['stats']['retweets'])
                        data['comments'].append(tweet['stats']['comments'])

                # Save data incrementally
                df = pd.DataFrame(data)
                df.to_csv('road_safety_combination_tweets.csv', index=False)
                print(f"Tweets for query '{query}' saved successfully.")
                instance_failed = False  # Mark as successful
                break  # Exit instance loop if successful

            else:
                print(f"No tweets found for query: {query}")
                instance_failed = True

        except Exception as e:
            print(f"Failed for query: {query} on instance {instance} due to {e}")
            instance_failed = True
            random_sleep(5, 10)  # Sleep longer before retrying the next instance

    if instance_failed:
        failed_queries.append(query)

    random_sleep()  # Add random sleep to avoid being blocked

# Final saving and summary
if data['text']:
    df = pd.DataFrame(data)
    df.to_csv('road_safety_combination_tweets.csv', index=False)
    print("Final tweets saved to road_safety_combination_tweets.csv.")
else:
    print("No tweets were collected.")

if failed_queries:
    print("Retry the following queries due to issues:", failed_queries)
else:
    print("All queries processed successfully.")


Testing instances: 100%|██████████| 10/10 [00:14<00:00,  1.47s/it]
INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Switched to Nitter instance: https://nitter.net
Switched to Nitter instance: https://nitter.net


Found tweets for query: Ferozepur Road accident
Tweets for query 'Ferozepur Road accident' saved successfully.


INFO:root:No instance specified, using random instance https://nitter.privacydev.net


Switched to Nitter instance: https://nitter.net


...

Switched to Nitter instance: https://nitter.privacydev.net
Failed for query: Korangi Road روڈ بلاک on instance https://nitter.privacydev.net due to list index out of range


INFO:root:No instance specified, using random instance https://lightbrd.com


Switched to Nitter instance: https://lightbrd.com
Failed for query: Korangi Road روڈ بلاک on instance https://lightbrd.com due to list index out of range


INFO:root:No instance specified, using random instance https://lightbrd.com


Switched to Nitter instance: https://nitter.net
Failed for query: Korangi Road ٹریفک جام on instance https://nitter.net due to list index out of range
